# Construction of Daily Geopolitical Risk Indices

<p><i>Author:</i> Angelica Vanti</p>

<p><i>Project:</i> Predicting EUR/USD Movements Using Geopolitical and Macroeconomic Variables</p>

This notebook transforms the final tweet-level LLM geopolitical-risk scores into daily time-series indices for subsequent econometric modelling.

Three geopolitical dimensions are considered: **Trade Hostility**, **Sanctions Threat**, and **Federal Reserve Pressure**. Alternative daily aggregation methods are constructed to examine whether different representations of geopolitical information provide greater predictive value for EUR/USD movements.

The notebook produces daily maximum and daily summed scores, together with 3-, 5-, and 10-day moving averages of the summed indices.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [11]:
# Repository paths
ROOT = Path("..")
PROCESSED_DIR = ROOT / "data" / "processed"

# Input: final truths annotations for validation
ANNOTATIONS_PATH = (
    PROCESSED_DIR
    / "truths_deepseek_outputs.csv"
)

# Outputs
DAILY_MAX_PATH = (
    PROCESSED_DIR
    / "truths_daily_max_deepseek.csv"
)

DAILY_SUM_PATH = (
    PROCESSED_DIR
    / "truths_daily_sum_deepseek.csv"
)

DAILY_MA3_PATH = (
    PROCESSED_DIR
    / "truths_daily_sum_ma3_deepseek.csv"
)

DAILY_MA5_PATH = (
    PROCESSED_DIR
    / "truths_daily_sum_ma5_deepseek.csv"
)

DAILY_MA10_PATH = (
    PROCESSED_DIR
    / "truths_daily_sum_ma10_deepseek.csv"
)

In [3]:
tweets = pd.read_csv(ANNOTATIONS_PATH)

print(f"Rows loaded: {len(tweets):,}")
print("\nColumns:")
print(tweets.columns.tolist())

print("\nFirst 5 rows:")
display(tweets.head())

print("\nMissing values:")
print(tweets.isna().sum())

Rows loaded: 5,176

Columns:
['tweet_index', 'tweet_id', 'date', 'tweet_snippet', 'trade_score', 'sanctions_score', 'fed_pressure_score', 'reasoning', 'source', 'example_ids', 'model', 'prompt_version', 'created_at_utc']

First 5 rows:


,tweet_index,tweet_id,date,tweet_snippet,trade_score,sanctions_score,fed_pressure_score,reasoning,source,example_ids,model,prompt_version,created_at_utc
0,0,113404826487996526,2024-11-01 00:18:46,Kamala has spent the final week of her failing...,0,0,0,The tweet does not mention anything about the ...,llm:native_structured_output,NaN,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-08-21T00:26:32.881126+00:00
1,1,113404838425751868,2024-11-01 00:21:48,"Just days ago, a young USMC veteran named Nich...",20,0,0,The tweet does not mention the Federal Reserve...,llm:native_structured_output,NaN,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-08-21T00:27:12.414588+00:00
2,2,113404894982236716,2024-11-01 00:36:11,"GET OUT AND VOTE, NEVADA!!!NEVADA VOTING INFOR...",0,0,0,The tweet does not mention anything related to...,llm:native_structured_output,NaN,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-08-21T00:27:20.353576+00:00
3,3,113405441290422074,2024-11-01 02:55:07,It was hardworking Patriots like you who built...,0,0,0,The tweet does not mention the Federal Reserve...,llm:native_structured_output,NaN,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-08-21T00:27:28.870399+00:00
4,4,113405998200625584,2024-11-01 05:16:44,"A GREAT DAY IN NEW MEXICO, NEVADA, AND ARIZONA...",0,0,0,The tweet does not mention anything related to...,llm:native_structured_output,NaN,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-08-21T00:27:35.707895+00:00



Missing values:
tweet_index              0
tweet_id                 0
date                     0
tweet_snippet            0
trade_score              0
sanctions_score          0
fed_pressure_score       0
reasoning                0
source                   0
example_ids           5176
model                    0
prompt_version           0
created_at_utc           0
dtype: int64


In [4]:
# Columns used to construct the geopolitical indices
SCORE_COLS = [
    "trade_score",
    "sanctions_score",
    "fed_pressure_score",
]

# Convert date to datetime
tweets["date"] = pd.to_datetime(tweets["date"], errors="coerce")

# Check that dates converted successfully
assert tweets["date"].notna().all(), "Some dates could not be parsed."

# Check that scores are present and within the expected 0-100 range
assert tweets[SCORE_COLS].notna().all().all(), "Missing geopolitical scores found."

for col in SCORE_COLS:
    assert tweets[col].between(0, 100).all(), f"{col} contains values outside 0-100."

# Sort chronologically
tweets = tweets.sort_values("date").reset_index(drop=True)

print("Date range:")
print(tweets["date"].min(), "to", tweets["date"].max())

print("\nScore ranges:")
print(tweets[SCORE_COLS].agg(["min", "max"]))

print("\nTweets per day:")
print(tweets.groupby(tweets["date"].dt.date).size().describe())

Date range:
2024-11-01 00:18:46 to 2026-06-11 01:21:45

Score ranges:
     trade_score  sanctions_score  fed_pressure_score
min            0                0                   0
max          100               90                 100

Tweets per day:
count    576.000000
mean       8.986111
std        8.087767
min        1.000000
25%        4.750000
50%        8.000000
75%       11.000000
max      116.000000
dtype: float64


## 1. Daily Aggregation of Tweet-Level Scores

Tweet-level geopolitical scores are aggregated by calendar day using two alternative measures.

The **daily maximum** records the highest score observed within each geopolitical category on a given day. This measure captures the intensity of the most geopolitically significant tweet posted that day.

The **daily sum** adds the scores of all tweets posted on each day. This measure captures both the intensity and frequency of geopolitical content and therefore allows multiple relevant tweets on the same day to contribute to the daily index.

In [5]:
# Create calendar-day variable while preserving original timestamp
tweets["calendar_date"] = tweets["date"].dt.normalize()

# Daily maximum score
daily_max = (
    tweets
    .groupby("calendar_date")[SCORE_COLS]
    .max()
    .reset_index()
)

daily_max = daily_max.rename(columns={
    "calendar_date": "date",
    "trade_score": "trade_max",
    "sanctions_score": "sanctions_max",
    "fed_pressure_score": "fed_pressure_max",
})

# Daily sum of scores
daily_sum = (
    tweets
    .groupby("calendar_date")[SCORE_COLS]
    .sum()
    .reset_index()
)

daily_sum = daily_sum.rename(columns={
    "calendar_date": "date",
    "trade_score": "trade_sum",
    "sanctions_score": "sanctions_sum",
    "fed_pressure_score": "fed_pressure_sum",
})

print("Tweeting days:", len(daily_sum))

print("\nDaily maximum:")
display(daily_max.head())

print("\nDaily sum:")
display(daily_sum.head())

Tweeting days: 576

Daily maximum:


,date,trade_max,sanctions_max,fed_pressure_max
0,2024-11-01,45,0,0
1,2024-11-02,85,0,0
2,2024-11-03,0,0,0
3,2024-11-04,0,40,0
4,2024-11-05,40,0,0



Daily sum:


,date,trade_sum,sanctions_sum,fed_pressure_sum
0,2024-11-01,110,0,0
1,2024-11-02,85,0,0
2,2024-11-03,0,0,0
3,2024-11-04,0,40,0
4,2024-11-05,40,0,0


## 2. Construction of a Complete Daily Calendar

A complete calendar is constructed from the first to the final date in the tweet sample so that days on which no tweets were posted remain explicitly represented in the geopolitical time series.

For days without tweets, the geopolitical indices are assigned a value of zero. This represents the absence of a new tweet-based geopolitical signal rather than treating the observation as missing data.

This produces a continuous daily series suitable for alignment with the macroeconomic dataset.

In [6]:
# Complete calendar from first to last day in the sample
full_calendar = pd.DataFrame({
    "date": pd.date_range(
        start=tweets["calendar_date"].min(),
        end=tweets["calendar_date"].max(),
        freq="D"
    )
})

# Identify calendar days on which there were no tweets
no_tweet_days = full_calendar.loc[
    ~full_calendar["date"].isin(daily_sum["date"]),
    "date"
]

print(f"Total calendar days: {len(full_calendar)}")
print(f"Days with tweets: {len(daily_sum)}")
print(f"Days without tweets: {len(no_tweet_days)}")

print("\nDates with no tweets:")
print(no_tweet_days.to_string(index=False))

Total calendar days: 588
Days with tweets: 576
Days without tweets: 12

Dates with no tweets:
2024-11-07
2024-11-15
2024-11-22
2025-01-20
2025-09-06
2025-10-28
2025-11-13
2025-11-21
2025-12-21
2025-12-29
2026-01-04
2026-06-07


In [8]:
# Merge daily maximum onto complete calendar
daily_max = full_calendar.merge(
    daily_max,
    on="date",
    how="left"
)

# Merge daily sum onto complete calendar
daily_sum = full_calendar.merge(
    daily_sum,
    on="date",
    how="left"
)

# No tweet = no new geopolitical signal
daily_max[
    ["trade_max", "sanctions_max", "fed_pressure_max"]
] = daily_max[
    ["trade_max", "sanctions_max", "fed_pressure_max"]
].fillna(0)

daily_sum[
    ["trade_sum", "sanctions_sum", "fed_pressure_sum"]
] = daily_sum[
    ["trade_sum", "sanctions_sum", "fed_pressure_sum"]
].fillna(0)

print("Daily max rows:", len(daily_max))
print("Daily sum rows:", len(daily_sum))

print("\nMissing values in daily max:")
print(daily_max.isna().sum())

print("\nMissing values in daily sum:")
print(daily_sum.isna().sum())

print("\nExample no-tweet day:")
display(
    daily_sum[daily_sum["date"] == "2026-06-07"]
)

Daily max rows: 588
Daily sum rows: 588

Missing values in daily max:
date                0
trade_max           0
sanctions_max       0
fed_pressure_max    0
dtype: int64

Missing values in daily sum:
date                0
trade_sum           0
sanctions_sum       0
fed_pressure_sum    0
dtype: int64

Example no-tweet day:


,date,trade_sum,sanctions_sum,fed_pressure_sum
583,2026-06-07,0.0,0.0,0.0


## 3. Moving-Average Geopolitical Indices

To examine whether geopolitical information has a more persistent effect than a single-day measure can capture, moving averages of the daily summed indices are constructed over **3-, 5-, and 10-day windows**.

These measures smooth short-term fluctuations while allowing geopolitical signals to persist across subsequent days. Moving averages are calculated only when a complete rolling window is available, so the initial observations remain missing where insufficient historical data exists.

In [9]:
# Columns containing the daily summed geopolitical scores
SUM_COLS = [
    "trade_sum",
    "sanctions_sum",
    "fed_pressure_sum",
]

# Moving-average windows
windows = [3, 5, 10]

# Store each moving-average dataset
moving_averages = {}

for window in windows:
    ma = daily_sum[["date"]].copy()

    for col in SUM_COLS:
        ma[f"{col}_ma{window}"] = (
            daily_sum[col]
            .rolling(window=window, min_periods=window)
            .mean()
        )

    moving_averages[window] = ma

daily_ma3 = moving_averages[3]
daily_ma5 = moving_averages[5]
daily_ma10 = moving_averages[10]

print("MA(3):")
display(daily_ma3.head(12))

print("\nMA(5):")
display(daily_ma5.head(12))

print("\nMA(10):")
display(daily_ma10.head(12))

MA(3):


,date,trade_sum_ma3,sanctions_sum_ma3,fed_pressure_sum_ma3
0,2024-11-01,NaN,NaN,NaN
1,2024-11-02,NaN,NaN,NaN
2,2024-11-03,65.000000,0.000000,0.0
3,2024-11-04,28.333333,13.333333,0.0
4,2024-11-05,13.333333,13.333333,0.0
5,2024-11-06,13.333333,13.333333,0.0
6,2024-11-07,13.333333,0.000000,0.0
7,2024-11-08,13.333333,10.000000,0.0
8,2024-11-09,13.333333,10.000000,0.0
9,2024-11-10,13.333333,10.000000,0.0



MA(5):


,date,trade_sum_ma5,sanctions_sum_ma5,fed_pressure_sum_ma5
0,2024-11-01,NaN,NaN,NaN
1,2024-11-02,NaN,NaN,NaN
2,2024-11-03,NaN,NaN,NaN
3,2024-11-04,NaN,NaN,NaN
4,2024-11-05,47.0,8.0,0.0
5,2024-11-06,25.0,8.0,0.0
6,2024-11-07,8.0,8.0,0.0
7,2024-11-08,16.0,14.0,0.0
8,2024-11-09,16.0,6.0,0.0
9,2024-11-10,8.0,6.0,0.0



MA(10):


,date,trade_sum_ma10,sanctions_sum_ma10,fed_pressure_sum_ma10
0,2024-11-01,NaN,NaN,NaN
1,2024-11-02,NaN,NaN,NaN
2,2024-11-03,NaN,NaN,NaN
3,2024-11-04,NaN,NaN,NaN
4,2024-11-05,NaN,NaN,NaN
5,2024-11-06,NaN,NaN,NaN
6,2024-11-07,NaN,NaN,NaN
7,2024-11-08,NaN,NaN,NaN
8,2024-11-09,NaN,NaN,NaN
9,2024-11-10,27.5,7.0,0.0


## 4. Validation and Export

Before export, the constructed indices are validated to confirm the expected sample length, absence of unintended missing values, consistency between the maximum and summed measures, and the expected treatment of incomplete moving-average windows.

The five resulting geopolitical datasets are then exported for subsequent alignment with the macroeconomic variables and VAR-X modelling.

In [10]:
# Moving averages should only have missing values
# where a complete rolling window does not yet exist

# MA(3): first 2 observations undefined, everything afterwards defined
assert daily_ma3.iloc[:2, 1:].isna().all().all()
assert daily_ma3.iloc[2:, 1:].notna().all().all()

# MA(5): first 4 observations undefined, everything afterwards defined
assert daily_ma5.iloc[:4, 1:].isna().all().all()
assert daily_ma5.iloc[4:, 1:].notna().all().all()

# MA(10): first 9 observations undefined, everything afterwards defined
assert daily_ma10.iloc[:9, 1:].isna().all().all()
assert daily_ma10.iloc[9:, 1:].notna().all().all()

print("Daily MAX:   ", daily_max.shape)
print("Daily SUM:   ", daily_sum.shape)
print("SUM MA(3):   ", daily_ma3.shape)
print("SUM MA(5):   ", daily_ma5.shape)
print("SUM MA(10):  ", daily_ma10.shape)

Daily MAX:    (588, 4)
Daily SUM:    (588, 4)
SUM MA(3):    (588, 4)
SUM MA(5):    (588, 4)
SUM MA(10):   (588, 4)


In [12]:
# Save daily maximum and daily sum indices
daily_max.to_csv(DAILY_MAX_PATH, index=False)
daily_sum.to_csv(DAILY_SUM_PATH, index=False)

# Save moving-average versions of the daily sum
daily_ma3.to_csv(DAILY_MA3_PATH, index=False)
daily_ma5.to_csv(DAILY_MA5_PATH, index=False)
daily_ma10.to_csv(DAILY_MA10_PATH, index=False)

print("Saved:")
print(DAILY_MAX_PATH)
print(DAILY_SUM_PATH)
print(DAILY_MA3_PATH)
print(DAILY_MA5_PATH)
print(DAILY_MA10_PATH)

Saved:
..\data\processed\truths_daily_max_deepseek.csv
..\data\processed\truths_daily_sum_deepseek.csv
..\data\processed\truths_daily_sum_ma3_deepseek.csv
..\data\processed\truths_daily_sum_ma5_deepseek.csv
..\data\processed\truths_daily_sum_ma10_deepseek.csv


In [13]:
saved_files = {
    "truths_daily_max": DAILY_MAX_PATH,
    "truths_daily_sum": DAILY_SUM_PATH,
    "truths_daily_ma3": DAILY_MA3_PATH,
    "truths_daily_ma5": DAILY_MA5_PATH,
    "truths_daily_ma10": DAILY_MA10_PATH,
}

for name, path in saved_files.items():
    df = pd.read_csv(path)

    print(
        f"{name}: "
        f"{df.shape[0]} rows, "
        f"{df.shape[1]} columns"
    )

truths_daily_max: 588 rows, 4 columns
truths_daily_sum: 588 rows, 4 columns
truths_daily_ma3: 588 rows, 4 columns
truths_daily_ma5: 588 rows, 4 columns
truths_daily_ma10: 588 rows, 4 columns


## 5. Summary

Tweet-level LLM geopolitical-risk scores were transformed into five alternative daily representations for each of the Trade Hostility, Sanctions Threat and Federal Reserve Pressure indices.

The resulting datasets comprise a daily maximum measure, a daily summed measure, and 3-, 5-, and 10-day moving averages of the summed scores. A complete calendar was retained throughout the sample, with no-tweet days assigned zero geopolitical signal.

These alternative specifications allow the subsequent modelling analysis to assess whether EUR/USD dynamics are better explained by the most intense daily geopolitical event, the cumulative amount of geopolitical content, or a smoothed measure capturing persistence across several days.